In [ ]:
# @title Configuração de Ambiente
!pip -q install scikit-learn deap pandas seaborn matplotlib numpy

import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Callable, Optional, Any
from dataclasses import dataclass, asdict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from deap import base, creator, tools, algorithms
from collections import defaultdict

# Configuração para forçar CPU e silenciar logs
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

# Seeds para reprodutibilidade
random.seed(42)
np.random.seed(42)

# Configuração de estilo para visualizações
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso!')
print('📦 Bibliotecas: scikit-learn, DEAP, pandas, seaborn, matplotlib, numpy')
# Saída Esperada: ✅ Ambiente configurado com sucesso!

# Workshop Prático: Auditoria Ética de Sistemas de Otimização

## Contexto e Objetivo

Neste laboratório prático, você implementará um **sistema completo de auditoria ética** para analisar e mitigar problemas em sistemas de otimização. O caso de estudo é realista: um sistema de recomendação de vagas de emprego que, apesar de aparentar ser neutro, pode perpetuar vieses históricos.

## O que você vai aprender:

1. **Demonstrar a Lei de Goodhart na prática:** Como diferentes métricas de otimização levam a comportamentos problemáticos
2. **Implementar métricas de fairness:** Demographic Parity, Equalized Odds, Individual Fairness e Counterfactual Fairness
3. **Realizar Red Team Exercises:** Testar adversarialmente sistemas para encontrar vulnerabilidades éticas
4. **Projetar mitigações:** Implementar três abordagens para tornar sistemas mais éticos
5. **Analisar trade-offs:** Entender os custos de performance vs. fairness

## Estrutura do Workshop (210 min)

- **Parte 1:** Setup do sistema de recomendação (30 min)
- **Parte 2:** Análise da Lei de Goodhart (30 min)
- **Parte 3:** Framework de auditoria ética (30 min)
- **Parte 4:** Métricas de fairness (30 min)
- **Parte 5:** Red team exercises (30 min)
- **Parte 6:** Design de mitigações (30 min)
- **Parte 7:** Análise de trade-offs (20 min)
- **Parte 8:** Relatório automatizado (10 min)

In [ ]:
# @title Definição de Estruturas de Dados

@dataclass
class Candidato:
    """Representa um candidato no sistema de recomendação.
    
    Attributes:
        id: Identificador único
        experiencia_anos: Anos de experiência profissional
        educacao: Nível educacional (0-4)
        habilidades_tecnicas: Score 0-100
        habilidades_soft: Score 0-100
        genero: 'M' ou 'F' (atributo protegido)
        etnia: Categoria étnica (atributo protegido)
        idade: Idade em anos
    """
    id: int
    experiencia_anos: float
    educacao: int  # 0: Fund, 1: Médio, 2: Superior, 3: Mestrado, 4: Doutorado
    habilidades_tecnicas: float
    habilidades_soft: float
    genero: str  # Atributo protegido
    etnia: str   # Atributo protegido
    idade: int

@dataclass
class Vaga:
    """Representa uma vaga de emprego.
    
    Attributes:
        id: Identificador único
        requisito_exp_min: Anos mínimos de experiência
        requisito_educacao: Nível educacional mínimo
        requisito_habilidades_tecnicas: Score mínimo
        salario: Faixa salarial oferecida
    """
    id: int
    requisito_exp_min: float
    requisito_educacao: int
    requisito_habilidades_tecnicas: float
    salario: float

@dataclass
class ResultadoRecomendacao:
    """Resultado de uma recomendação."""
    candidato_id: int
    vaga_id: int
    score: float
    recomendado: bool

print('✅ Estruturas definidas: Candidato, Vaga, ResultadoRecomendacao')

In [ ]:
# @title Parte 1: Geração de Dataset com Vieses Históricos

def gerar_candidatos_com_vieses(n: int = 1000) -> List[Candidato]:
    """Gera candidatos sintéticos com vieses históricos implícitos.
    
    Os vieses simulam padrões históricos reais:
    - Mulheres têm, em média, menos anos de experiência devido a gaps de carreira
    - Pessoas negras têm, em média, menor acesso a educação superior
    - Habilidades correlacionadas com oportunidades desiguais
    
    Parameters:
        n: Número de candidatos a gerar
    
    Returns:
        Lista de objetos Candidato
    """
    candidatos = []
    
    for i in range(n):
        # Atributos protegidos gerados primeiro
        genero = np.random.choice(['M', 'F'], p=[0.5, 0.5])
        etnia = np.random.choice(
            ['Branca', 'Parda', 'Preta', 'Amarela', 'Indigena'],
            p=[0.45, 0.35, 0.15, 0.04, 0.01]
        )
        
        idade = np.random.randint(22, 60)
        
        # VIÉS 1: Experiência afetada por gênero (gaps de carreira)
        exp_base = max(0, idade - 22)
        if genero == 'F':
            # Mulheres: -15% experiência média (gaps de carreira)
            experiencia_anos = max(0, exp_base * 0.85 + np.random.normal(0, 1))
        else:
            experiencia_anos = max(0, exp_base + np.random.normal(0, 1))
        
        # VIÉS 2: Educação afetada por etnia
        if etnia in ['Preta', 'Indigena']:
            educacao = np.random.choice([0, 1, 2, 3, 4], p=[0.1, 0.3, 0.45, 0.1, 0.05])
        elif etnia == 'Parda':
            educacao = np.random.choice([0, 1, 2, 3, 4], p=[0.05, 0.25, 0.50, 0.15, 0.05])
        else:  # Branca, Amarela
            educacao = np.random.choice([0, 1, 2, 3, 4], p=[0.02, 0.15, 0.50, 0.25, 0.08])
        
        # Habilidades técnicas correlacionadas com educação e experiência
        hab_tec_base = (educacao * 15) + (experiencia_anos * 2) + np.random.normal(30, 10)
        habilidades_tecnicas = np.clip(hab_tec_base, 0, 100)
        
        # Habilidades soft: menos enviesadas
        habilidades_soft = np.clip(np.random.normal(60, 15), 0, 100)
        
        candidato = Candidato(
            id=i,
            experiencia_anos=round(experiencia_anos, 1),
            educacao=educacao,
            habilidades_tecnicas=round(habilidades_tecnicas, 1),
            habilidades_soft=round(habilidades_soft, 1),
            genero=genero,
            etnia=etnia,
            idade=idade
        )
        candidatos.append(candidato)
    
    return candidatos

def gerar_vagas(n: int = 20) -> List[Vaga]:
    """Gera vagas de emprego com requisitos variados."""
    vagas = []
    for i in range(n):
        nivel = np.random.choice(['junior', 'pleno', 'senior'], p=[0.4, 0.4, 0.2])
        
        if nivel == 'junior':
            exp_min = np.random.uniform(0, 2)
            educacao_min = np.random.choice([1, 2], p=[0.3, 0.7])
            hab_tec_min = np.random.uniform(40, 60)
            salario = np.random.uniform(3000, 6000)
        elif nivel == 'pleno':
            exp_min = np.random.uniform(2, 5)
            educacao_min = np.random.choice([2, 3], p=[0.7, 0.3])
            hab_tec_min = np.random.uniform(60, 80)
            salario = np.random.uniform(6000, 12000)
        else:  # senior
            exp_min = np.random.uniform(5, 10)
            educacao_min = np.random.choice([2, 3, 4], p=[0.5, 0.4, 0.1])
            hab_tec_min = np.random.uniform(75, 95)
            salario = np.random.uniform(12000, 25000)
        
        vaga = Vaga(
            id=i,
            requisito_exp_min=round(exp_min, 1),
            requisito_educacao=educacao_min,
            requisito_habilidades_tecnicas=round(hab_tec_min, 1),
            salario=round(salario, 2)
        )
        vagas.append(vaga)
    
    return vagas

# Gerar dados
candidatos = gerar_candidatos_com_vieses(1000)
vagas = gerar_vagas(20)

# Converter para DataFrame para análise
df_candidatos = pd.DataFrame([asdict(c) for c in candidatos])
df_vagas = pd.DataFrame([asdict(v) for v in vagas])

print(f'✅ Gerados {len(candidatos)} candidatos e {len(vagas)} vagas')
print(f'\n📊 Distribuição por gênero:')
print(df_candidatos['genero'].value_counts())
print(f'\n📊 Distribuição por etnia:')
print(df_candidatos['etnia'].value_counts())
print(f'\n📊 Experiência média por gênero (evidência de viés):')
print(df_candidatos.groupby('genero')['experiencia_anos'].mean().round(2))
print(f'\n📊 Educação superior (>=2) por etnia:')
print((df_candidatos.groupby('etnia')['educacao'].apply(lambda x: (x >= 2).mean()) * 100).round(1))
# Saída Esperada: Estatísticas evidenciando vieses históricos nos dados

In [ ]:
# @title Parte 2: Demonstração da Lei de Goodhart

def fitness_cliques(candidato: Candidato, vaga: Vaga) -> float:
    """Métrica 1: Maximizar cliques.
    
    PROBLEMA: Candidatos MENOS qualificados clicam MAIS (desespero).
    Otimizar para cliques incentiva recomendar para quem NÃO deveria.
    """
    gap_exp = max(0, vaga.requisito_exp_min - candidato.experiencia_anos)
    gap_edu = max(0, vaga.requisito_educacao - candidato.educacao)
    gap_hab = max(0, vaga.requisito_habilidades_tecnicas - candidato.habilidades_tecnicas)
    
    # Quanto MAIOR o gap, MAIOR o score (perverso!)
    return min(1.0, (gap_exp * 0.3 + gap_edu * 10 + gap_hab * 0.5) / 50)

def fitness_contratacao(candidato: Candidato, vaga: Vaga) -> float:
    """Métrica 2: Maximizar taxa de contratação.
    
    PROBLEMA: Incentiva recomendar apenas overqualified (aposta segura),
    perpetuando desigualdades.
    """
    bonus_exp = max(0, candidato.experiencia_anos - vaga.requisito_exp_min)
    bonus_edu = max(0, candidato.educacao - vaga.requisito_educacao)
    bonus_hab = max(0, candidato.habilidades_tecnicas - vaga.requisito_habilidades_tecnicas)
    
    return min(1.0, 0.5 + (bonus_exp * 0.3 + bonus_edu * 10 + bonus_hab * 0.5) / 50)

def fitness_fit_cultural(candidato: Candidato, vaga: Vaga) -> float:
    """Métrica 3: 'Fit cultural' baseado em histórico.
    
    PROBLEMA: 'Fit cultural' é proxy para 'similar a nós',
    perpetuando homogeneidade via vieses implícitos.
    """
    score = 0.5
    
    # VIÉS: 'fit' favorece grupos majoritários
    if candidato.genero == 'M':
        score += 0.2
    if candidato.etnia == 'Branca':
        score += 0.2
    
    score += candidato.habilidades_soft / 100 * 0.3
    return min(1.0, score)

def fitness_balanced(candidato: Candidato, vaga: Vaga) -> float:
    """Função balanceada (baseline)."""
    score_exp = 1.0 if candidato.experiencia_anos >= vaga.requisito_exp_min else candidato.experiencia_anos / vaga.requisito_exp_min
    score_edu = 1.0 if candidato.educacao >= vaga.requisito_educacao else 0.5
    score_hab = 1.0 if candidato.habilidades_tecnicas >= vaga.requisito_habilidades_tecnicas else candidato.habilidades_tecnicas / vaga.requisito_habilidades_tecnicas
    
    return 0.3 * score_exp + 0.3 * score_edu + 0.4 * score_hab

# Demonstração comparativa
print('🔬 Lei de Goodhart: Diferentes métricas → Diferentes comportamentos\n')

vaga_teste = [v for v in vagas if 4 < v.requisito_exp_min < 6][0]
cand_under = [c for c in candidatos if c.experiencia_anos < 2][0]
cand_qualified = [c for c in candidatos if 3 < c.experiencia_anos < 5][0]
cand_over = [c for c in candidatos if c.experiencia_anos > 10][0]

resultados = []
for nome, cand in [('Underqualified', cand_under),
                    ('Qualified', cand_qualified),
                    ('Overqualified', cand_over)]:
    resultados.append({
        'Candidato': nome,
        'Exp': cand.experiencia_anos,
        'Cliques': round(fitness_cliques(cand, vaga_teste), 3),
        'Contratação': round(fitness_contratacao(cand, vaga_teste), 3),
        'Fit Cultural': round(fitness_fit_cultural(cand, vaga_teste), 3),
        'Balanced': round(fitness_balanced(cand, vaga_teste), 3)
    })

print(f'Vaga: {vaga_teste.requisito_exp_min} anos exp mín.')
print(pd.DataFrame(resultados).to_string(index=False))
print('\n⚠️ OBSERVE os incentivos perversos de cada métrica!')
# Saída Esperada: Tabela mostrando comportamentos contraditórios por métrica

In [ ]:
# @title Parte 3: Implementação de Métricas de Fairness

def calcular_demographic_parity(recomendacoes: List[ResultadoRecomendacao], 
                                candidatos: List[Candidato],
                                atributo_protegido: str) -> Dict[str, float]:
    """Calcula Demographic Parity: taxa de recomendações deve ser igual entre grupos.
    
    Parameters:
        recomendacoes: Lista de resultados
        candidatos: Lista de candidatos
        atributo_protegido: 'genero' ou 'etnia'
    
    Returns:
        Dicionário com taxas por grupo
    """
    cand_map = {c.id: c for c in candidatos}
    grupos = defaultdict(lambda: {'total': 0, 'recomendados': 0})
    
    for rec in recomendacoes:
        cand = cand_map[rec.candidato_id]
        grupo = getattr(cand, atributo_protegido)
        grupos[grupo]['total'] += 1
        if rec.recomendado:
            grupos[grupo]['recomendados'] += 1
    
    taxas = {}
    for grupo, stats in grupos.items():
        taxas[grupo] = stats['recomendados'] / stats['total'] if stats['total'] > 0 else 0
    
    return taxas

def calcular_disparate_impact(taxas: Dict[str, float]) -> float:
    """Calcula disparate impact: razão entre menor e maior taxa.
    
    Valor >= 0.8 é considerado aceitável pela regra dos 80%.
    
    Returns:
        Valor entre 0 e 1 (1 = perfeita paridade)
    """
    if not taxas or len(taxas) < 2:
        return 1.0
    
    valores = list(taxas.values())
    min_taxa = min(valores)
    max_taxa = max(valores)
    
    if max_taxa == 0:
        return 1.0
    
    return min_taxa / max_taxa

def calcular_individual_fairness(candidato1: Candidato, candidato2: Candidato,
                                 score1: float, score2: float) -> float:
    """Calcula Individual Fairness: candidatos similares devem ter scores similares.
    
    Usa distância euclidiana normalizada de atributos não-protegidos.
    
    Returns:
        Diferença entre similaridade e diferença de scores (0 = perfeitamente fair)
    """
    # Similaridade baseada em atributos não-protegidos
    diff_exp = abs(candidato1.experiencia_anos - candidato2.experiencia_anos) / 40  # normalizado por max
    diff_edu = abs(candidato1.educacao - candidato2.educacao) / 4
    diff_hab_tec = abs(candidato1.habilidades_tecnicas - candidato2.habilidades_tecnicas) / 100
    diff_hab_soft = abs(candidato1.habilidades_soft - candidato2.habilidades_soft) / 100
    
    similaridade = 1 - np.sqrt(diff_exp**2 + diff_edu**2 + diff_hab_tec**2 + diff_hab_soft**2) / 2
    diff_score = abs(score1 - score2)
    
    # Se candidatos são similares, scores devem ser similares
    return abs(similaridade - (1 - diff_score))

def calcular_counterfactual_fairness(candidato: Candidato, vaga: Vaga,
                                     fitness_func: Callable) -> float:
    """Calcula Counterfactual Fairness: score não deve mudar se mudarmos atributo protegido.
    
    Parameters:
        candidato: Candidato original
        vaga: Vaga avaliada
        fitness_func: Função de fitness a testar
    
    Returns:
        Diferença máxima de score entre contrafactuais (0 = perfeitamente fair)
    """
    score_original = fitness_func(candidato, vaga)
    scores = [score_original]
    
    # Criar contrafactuais variando gênero
    for genero in ['M', 'F']:
        cand_cf = Candidato(
            id=candidato.id,
            experiencia_anos=candidato.experiencia_anos,
            educacao=candidato.educacao,
            habilidades_tecnicas=candidato.habilidades_tecnicas,
            habilidades_soft=candidato.habilidades_soft,
            genero=genero,  # Variando
            etnia=candidato.etnia,
            idade=candidato.idade
        )
        scores.append(fitness_func(cand_cf, vaga))
    
    # Criar contrafactuais variando etnia
    for etnia in ['Branca', 'Parda', 'Preta']:
        cand_cf = Candidato(
            id=candidato.id,
            experiencia_anos=candidato.experiencia_anos,
            educacao=candidato.educacao,
            habilidades_tecnicas=candidato.habilidades_tecnicas,
            habilidades_soft=candidato.habilidades_soft,
            genero=candidato.genero,
            etnia=etnia,  # Variando
            idade=candidato.idade
        )
        scores.append(fitness_func(cand_cf, vaga))
    
    return max(scores) - min(scores)

print('✅ Métricas de fairness implementadas:')
print('  - Demographic Parity (paridade demográfica)')
print('  - Disparate Impact (impacto desproporcional)')
print('  - Individual Fairness (justiça individual)')
print('  - Counterfactual Fairness (justiça contrafactual)')

In [ ]:
# @title Parte 4: Framework de Auditoria Ética

def realizar_auditoria_etica(fitness_func: Callable,
                             candidatos: List[Candidato],
                             vagas: List[Vaga],
                             threshold: float = 0.5) -> Dict[str, Any]:
    """Realiza auditoria ética completa de uma função de fitness.
    
    Parameters:
        fitness_func: Função de fitness a auditar
        candidatos: Lista de candidatos
        vagas: Lista de vagas
        threshold: Limiar para considerar recomendado
    
    Returns:
        Relatório completo de auditoria
    """
    # Gerar recomendações
    recomendacoes = []
    for cand in candidatos[:100]:  # Sample para performance
        for vaga in vagas[:5]:  # Sample de vagas
            score = fitness_func(cand, vaga)
            recomendacoes.append(ResultadoRecomendacao(
                candidato_id=cand.id,
                vaga_id=vaga.id,
                score=score,
                recomendado=score >= threshold
            ))
    
    # Calcular métricas de fairness
    taxas_genero = calcular_demographic_parity(recomendacoes, candidatos, 'genero')
    taxas_etnia = calcular_demographic_parity(recomendacoes, candidatos, 'etnia')
    
    di_genero = calcular_disparate_impact(taxas_genero)
    di_etnia = calcular_disparate_impact(taxas_etnia)
    
    # Counterfactual fairness (sample)
    cf_scores = []
    for cand in candidatos[:20]:
        for vaga in vagas[:3]:
            cf_scores.append(calcular_counterfactual_fairness(cand, vaga, fitness_func))
    
    cf_media = np.mean(cf_scores)
    cf_max = np.max(cf_scores)
    
    relatorio = {
        'demographic_parity': {
            'genero': taxas_genero,
            'etnia': taxas_etnia
        },
        'disparate_impact': {
            'genero': di_genero,
            'etnia': di_etnia,
            'genero_aprovado': di_genero >= 0.8,
            'etnia_aprovado': di_etnia >= 0.8
        },
        'counterfactual_fairness': {
            'media': cf_media,
            'max': cf_max,
            'aprovado': cf_max < 0.1  # Diferença < 10% considerada aceitável
        },
        'total_recomendacoes': len([r for r in recomendacoes if r.recomendado]),
        'avaliacao_geral': 'APROVADO' if (di_genero >= 0.8 and di_etnia >= 0.8 and cf_max < 0.1) else 'REPROVADO'
    }
    
    return relatorio

print('✅ Framework de auditoria ética implementado')
print('\n🔍 Testando as funções de fitness problemáticas...\n')

# Auditar cada função
funcoes = [
    ('Cliques', fitness_cliques),
    ('Contratação', fitness_contratacao),
    ('Fit Cultural', fitness_fit_cultural),
    ('Balanced', fitness_balanced)
]

for nome, func in funcoes:
    print(f'\n{'='*60}')
    print(f'📋 Auditoria: {nome}')
    print('='*60)
    relatorio = realizar_auditoria_etica(func, candidatos, vagas)
    
    print(f"\nDisparate Impact (Gênero): {relatorio['disparate_impact']['genero']:.3f} {'✅' if relatorio['disparate_impact']['genero_aprovado'] else '❌'}")
    print(f"Disparate Impact (Etnia): {relatorio['disparate_impact']['etnia']:.3f} {'✅' if relatorio['disparate_impact']['etnia_aprovado'] else '❌'}")
    print(f"Counterfactual Fairness (max): {relatorio['counterfactual_fairness']['max']:.3f} {'✅' if relatorio['counterfactual_fairness']['aprovado'] else '❌'}")
    print(f"\n🏆 Avaliação Geral: {relatorio['avaliacao_geral']}")

print('\n⚠️ Observe como diferentes funções violam diferentes critérios de fairness!')
# Saída Esperada: Relatórios mostrando violações éticas em cada função

In [ ]:
# @title Parte 5: Red Team Exercise - Ataques Adversariais

def gaming_attack(fitness_func: Callable, vaga: Vaga, 
                  atributos_protegidos: Dict[str, str]) -> Candidato:
    """Ataque de gaming: criar perfil sintético que maximiza score injustamente.
    
    Usa algoritmo genético para encontrar candidato 'ótimo' com
    atributos protegidos fixos.
    
    Parameters:
        fitness_func: Função alvo
        vaga: Vaga alvo
        atributos_protegidos: Dict com genero e etnia fixos
    
    Returns:
        Candidato otimizado para 'gaming'
    """
    # Setup DEAP
    if hasattr(creator, 'FitnessMax'):
        del creator.FitnessMax
    if hasattr(creator, 'Individual'):
        del creator.Individual
    
    creator.create('FitnessMax', base.Fitness, weights=(1.0,))
    creator.create('Individual', list, fitness=creator.FitnessMax)
    
    toolbox = base.Toolbox()
    
    # Genes: [experiencia, educacao, hab_tec, hab_soft, idade]
    toolbox.register('exp', random.uniform, 0, 40)
    toolbox.register('edu', random.randint, 0, 4)
    toolbox.register('hab_tec', random.uniform, 0, 100)
    toolbox.register('hab_soft', random.uniform, 0, 100)
    toolbox.register('idade', random.randint, 22, 60)
    
    toolbox.register('individual', tools.initCycle, creator.Individual,
                     (toolbox.exp, toolbox.edu, toolbox.hab_tec, toolbox.hab_soft, toolbox.idade), n=1)
    toolbox.register('population', tools.initRepeat, list, toolbox.individual)
    
    def eval_gaming(individual):
        cand = Candidato(
            id=9999,
            experiencia_anos=individual[0],
            educacao=int(individual[1]),
            habilidades_tecnicas=individual[2],
            habilidades_soft=individual[3],
            genero=atributos_protegidos['genero'],
            etnia=atributos_protegidos['etnia'],
            idade=int(individual[4])
        )
        return (fitness_func(cand, vaga),)
    
    toolbox.register('evaluate', eval_gaming)
    toolbox.register('mate', tools.cxTwoPoint)
    toolbox.register('mutate', tools.mutGaussian, mu=0, sigma=5, indpb=0.2)
    toolbox.register('select', tools.selTournament, tournsize=3)
    
    # Evoluir
    pop = toolbox.population(n=50)
    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=20, verbose=False)
    
    # Melhor indivíduo
    best = tools.selBest(pop, 1)[0]
    return Candidato(
        id=9999,
        experiencia_anos=round(best[0], 1),
        educacao=int(best[1]),
        habilidades_tecnicas=round(best[2], 1),
        habilidades_soft=round(best[3], 1),
        genero=atributos_protegidos['genero'],
        etnia=atributos_protegidos['etnia'],
        idade=int(best[4])
    )

print('🚨 Red Team Exercise: Testando vulnerabilidades\n')

vaga_alvo = vagas[0]
print(f'Vaga alvo: {vaga_alvo.requisito_exp_min} anos, edu {vaga_alvo.requisito_educacao}')

# Testar gaming attack na função de 'Fit Cultural'
print('\n🎯 Gaming Attack na função "Fit Cultural" (enviesada)...')
perfil_gaming = gaming_attack(
    fitness_fit_cultural,
    vaga_alvo,
    {'genero': 'F', 'etnia': 'Preta'}  # Grupo desfavorecido
)

score_gaming = fitness_fit_cultural(perfil_gaming, vaga_alvo)
print(f'\nPerfil otimizado (gaming): Exp={perfil_gaming.experiencia_anos}, '
      f'Edu={perfil_gaming.educacao}, Hab Tec={perfil_gaming.habilidades_tecnicas:.1f}')
print(f'Score obtido: {score_gaming:.3f}')

# Comparar com candidato real equivalente
real_similar = [c for c in candidatos if c.genero == 'F' and c.etnia == 'Preta'][0]
score_real = fitness_fit_cultural(real_similar, vaga_alvo)
print(f'\nCandidato real similar: Score={score_real:.3f}')
print(f'\n⚠️ Ganho com gaming: {((score_gaming - score_real) / score_real * 100):.1f}%')

print('\n💡 Insight: Gaming attacks expõem como funções podem ser manipuladas!')
# Saída Esperada: Perfil sintético com score artificialmente inflado

In [ ]:
# @title Parte 6: Estratégias de Mitigação

def fitness_constraint_based(candidato: Candidato, vaga: Vaga) -> float:
    """Approach 1: Constraint-Based Optimization."""
    score = fitness_balanced(candidato, vaga)
    scores_cf = [score]
    for gen in ['M', 'F']:
        cand_cf = Candidato(
            candidato.id, candidato.experiencia_anos, candidato.educacao,
            candidato.habilidades_tecnicas, candidato.habilidades_soft,
            gen, candidato.etnia, candidato.idade
        )
        scores_cf.append(fitness_balanced(cand_cf, vaga))
    variacao = max(scores_cf) - min(scores_cf)
    if variacao > 0.1:
        score *= 0.5
    return score

def fitness_adversarial_debiasing(candidato: Candidato, vaga: Vaga) -> float:
    """Approach 2: Adversarial Debiasing - marginaliza atributos protegidos."""
    scores = []
    for gen in ['M', 'F']:
        for etn in ['Branca', 'Parda', 'Preta']:
            cand_debiased = Candidato(
                candidato.id, candidato.experiencia_anos, candidato.educacao,
                candidato.habilidades_tecnicas, candidato.habilidades_soft,
                gen, etn, candidato.idade
            )
            scores.append(fitness_balanced(cand_debiased, vaga))
    return np.mean(scores)

print('✅ Estratégias de mitigação implementadas')
cand_teste = candidatos[0]
vaga_teste = vagas[0]
print(f'\nTeste com {cand_teste.genero}, {cand_teste.etnia}:')
print(f'  Balanced:              {fitness_balanced(cand_teste, vaga_teste):.3f}')
print(f'  Constraint-Based:      {fitness_constraint_based(cand_teste, vaga_teste):.3f}')
print(f'  Adversarial Debiasing: {fitness_adversarial_debiasing(cand_teste, vaga_teste):.3f}')
# Saída Esperada: Scores mostrando efeito de cada mitigação

In [ ]:
# @title Parte 7: Análise de Trade-offs

def analisar_tradeoffs():
    abordagens = [
        ('Balanced', fitness_balanced),
        ('Fit Cultural (Enviesado)', fitness_fit_cultural),
        ('Constraint-Based', fitness_constraint_based),
        ('Adversarial Debiasing', fitness_adversarial_debiasing)
    ]
    resultados = []
    for nome, func in abordagens:
        print(f'Analisando: {nome}...')
        relatorio = realizar_auditoria_etica(func, candidatos[:100], vagas[:5])
        scores_perf = []
        for cand in candidatos[:50]:
            for vaga in vagas[:3]:
                scores_perf.append(func(cand, vaga))
        resultados.append({
            'Abordagem': nome,
            'DI Gênero': round(relatorio['disparate_impact']['genero'], 3),
            'DI Etnia': round(relatorio['disparate_impact']['etnia'], 3),
            'CF Max': round(relatorio['counterfactual_fairness']['max'], 3),
            'Performance': round(np.mean(scores_perf), 3),
            'Status': relatorio['avaliacao_geral']
        })
    df = pd.DataFrame(resultados)
    print('\n' + '='*70)
    print('ANÁLISE COMPARATIVA DE TRADE-OFFS')
    print('='*70)
    print(df.to_string(index=False))
    print('\n💡 Observe o trade-off fundamental entre performance e fairness!')
    return df

df_analise = analisar_tradeoffs()
# Saída Esperada: Tabela comparativa de todas as abordagens

In [ ]:
# @title Parte 8: Relatório de Auditoria Ética Automatizado

def gerar_relatorio_final(nome: str, fitness_func: Callable) -> str:
    rel = realizar_auditoria_etica(fitness_func, candidatos, vagas)
    
    md = f'''# RELATÓRIO DE AUDITORIA ÉTICA\n## {nome}\n\n'''
    md += f'''**Avaliação Geral:** {rel['avaliacao_geral']}\n\n'''
    md += f'''### Métricas de Fairness:\n'''
    md += f'''- Disparate Impact (Gênero): {rel['disparate_impact']['genero']:.3f}'''
    md += f''' ({'✅' if rel['disparate_impact']['genero_aprovado'] else '❌'})\n'''
    md += f'''- Disparate Impact (Etnia): {rel['disparate_impact']['etnia']:.3f}'''
    md += f''' ({'✅' if rel['disparate_impact']['etnia_aprovado'] else '❌'})\n'''
    md += f'''- Counterfactual Fairness: {rel['counterfactual_fairness']['max']:.3f}'''
    md += f''' ({'✅' if rel['counterfactual_fairness']['aprovado'] else '❌'})\n\n'''
    
    if rel['avaliacao_geral'] == 'REPROVADO':
        md += '''### Recomendações:\n'''
        md += '''1. Implementar constraint-based mitigation\n'''
        md += '''2. Considerar multi-objective optimization\n'''
        md += '''3. Aplicar adversarial debiasing\n'''
        md += '''4. Estabelecer monitoramento contínuo\n'''
    
    return md

print('📄 Gerando relatório de auditoria ética...\n')
relatorio = gerar_relatorio_final(
    'Sistema de Recomendação - Fit Cultural',
    fitness_fit_cultural
)
print(relatorio)
print('\n✅ Auditoria ética completa!')
# Saída Esperada: Relatório profissional em Markdown

## 🎓 Conclusões e Próximos Passos

### O que aprendemos:

1. **A Lei de Goodhart é real:** Métricas otimizadas podem ter consequências não intencionais
2. **Fairness é multidimensional:** Múltiplas métricas capturam aspectos diferentes
3. **Red team é essencial:** Testes adversariais revelam vulnerabilidades
4. **Mitigações têm custos:** Trade-off fundamental entre performance e fairness

### Checklist para seus projetos:

- [ ] Testar demographic parity e counterfactual fairness
- [ ] Realizar red team exercises
- [ ] Documentar trade-offs entre performance e ética
- [ ] Estabelecer monitoramento contínuo de fairness
- [ ] Agendar re-auditorias periódicas

### Recursos:

- **Livros:** "Fairness and Machine Learning" (fairmlbook.org)
- **Ferramentas:** AI Fairness 360 (IBM), Fairlearn (Microsoft)
- **Regulações:** EU AI Act, GDPR Article 22

### Mensagem final:

Como engenheiro de otimização, você tem poder sobre sistemas que afetam vidas.  
**Não basta não ser enviesado - é preciso ser anti-viés.**  
Construa sistemas que tornem o mundo melhor. 🚀

---

**Aula 14 - Workshop Prático de Ética em Otimização**  
**Duração:** 3.5 horas | **Nível:** Avançado